# Lesson 2: Database Modeling & SQL Mastery
### VisionStream — From CSV to Production PostgreSQL

---

## Learning Objectives

By the end of this lesson, you will be able to:

1. **Design** a relational database schema with proper relationships and constraints
2. **Write** DDL (Data Definition Language) SQL to create tables and indexes
3. **Explain** why indexes matter for query performance
4. **Write** a complex aggregation SQL query with GROUP BY and geographic filtering
5. **Import** hundreds of thousands of real GPS records into PostgreSQL using batch insert

---

## Why Not Just Use a CSV File?

| | CSV File | PostgreSQL |
|---|---|---|
| **Scale** | ~100k rows before slowdown | Billions of rows with proper indexes |
| **Queries** | Full file scan every time | Index-based lookup in milliseconds |
| **Concurrent writes** | Corrupts data | ACID transactions prevent corruption |
| **Relationships** | No enforcement | Foreign keys guarantee integrity |
| **Analytics** | Manual Python loops | SQL GROUP BY, aggregations, window functions |
| **Backup/Recovery** | Manual copy | Built-in WAL (Write-Ahead Log) |

For a system receiving 1,000 sensor readings per second from 1,000 helmets,
a CSV file would be overwritten and corrupted almost immediately.

## Section 1: Entity-Relationship Design

Before writing any SQL, draw the schema. Here is the VisionStream ER diagram:

```
┌─────────────────────────────────┐
│            devices              │
├─────────────────────────────────┤
│ id             UUID (PK)        │
│ device_id      VARCHAR UNIQUE   │◄──────┐
│ owner_name     VARCHAR          │       │ FK (1:N)
│ registered_at  TIMESTAMPTZ      │       │
│ firmware_version VARCHAR        │       │
│ is_active      BOOLEAN          │       │
└─────────────────────────────────┘       │
                                           │
        ┌──────────────────────────────────┤
        │                                  │
┌───────▼──────────────────────┐  ┌────────▼──────────────────────┐
│       telemetry_logs         │  │           alerts               │
├──────────────────────────────┤  ├───────────────────────────────┤
│ id          BIGSERIAL (PK)   │  │ id          BIGSERIAL (PK)    │
│ device_id   VARCHAR (FK)     │  │ device_id   VARCHAR (FK)      │
│ timestamp   TIMESTAMPTZ      │  │ timestamp   TIMESTAMPTZ       │
│ gps_lat     DOUBLE PRECISION │  │ alert_type  VARCHAR           │
│ gps_lon     DOUBLE PRECISION │  │ gps_lat     DOUBLE PRECISION  │
│ obstacle_cm INTEGER          │  │ gps_lon     DOUBLE PRECISION  │
│ alert_type  VARCHAR          │  │ severity    VARCHAR (CHECK)   │
│ battery     SMALLINT         │  │ resolved    BOOLEAN           │
└──────────────────────────────┘  └───────────────────────────────┘
```

### Relationships

- `devices` → `telemetry_logs` : **One-to-Many** (1 device, many log records)
- `devices` → `alerts`         : **One-to-Many** (1 device, many alert events)
- `device_id` is the foreign key that links these tables

### Key Design Decisions

| Decision | Reasoning |
|----------|----------|
| `UUID` for `devices.id` | Avoids primary key collisions when multiple API servers insert simultaneously (scale-out safety) |
| `BIGSERIAL` for logs/alerts | Supports up to 9.2×10¹⁸ rows; regular `SERIAL` maxes at ~2.1 billion |
| Separate `alerts` table | Telemetry logs are write-heavy; alert analytics are read-heavy. Separating them allows different optimization strategies |
| `TIMESTAMPTZ` not `TIMESTAMP` | Stores timezone-aware timestamps — critical when helmets are deployed globally |

## Section 2: Indexes — The Secret to Query Performance

An **index** is like a book's table of contents. Without it, PostgreSQL reads every row
to find matches (a *sequential scan*). With it, PostgreSQL jumps directly to the matching rows.

### Query Performance Comparison (hypothetical)

Query: Find all telemetry for device `helmet-001` in the last 24 hours.

| Without index | With composite index on `(device_id, timestamp DESC)` |
|---------------|-------------------------------------------------------|
| Scans all 500M rows | Skips directly to ~86,400 matching rows |
| ~30 seconds | ~5 milliseconds |
| Blocks other queries | Non-blocking |

### The Three Indexes You Need

```sql
-- Index 1: For GET /telemetry/{device_id}/history
-- Query pattern: WHERE device_id = ? ORDER BY timestamp DESC
CREATE INDEX idx_telemetry_device_time
    ON telemetry_logs (device_id, timestamp DESC);

-- Index 2: For GET /alerts/history?device_id=...
-- Query pattern: WHERE device_id = ? ORDER BY timestamp DESC
CREATE INDEX idx_alerts_device_time
    ON alerts (device_id, timestamp DESC);

-- Index 3: For GET /alerts/stats (the geo-grid query)
-- Query pattern: WHERE gps_lat BETWEEN ? AND ? AND gps_lon BETWEEN ? AND ?
CREATE INDEX idx_alerts_location
    ON alerts (gps_lat, gps_lon);
```

### When NOT to add an index
- Columns that are never filtered or sorted on
- Very small tables (< 1,000 rows) — sequential scan is faster
- Columns that are updated very frequently (indexes slow down writes)

## Section 3: Illustrative Code — SQL DDL Pattern

Study the pattern below. It shows the structure for `CREATE TABLE` with:
- Primary keys, foreign keys, and NOT NULL constraints
- A `CHECK` constraint that enforces data integrity at the database level
- `ON DELETE CASCADE` behavior

**Do NOT copy this directly.** Use it as a reference when writing `db/schema.sql`.

In [ ]:
# ── ILLUSTRATIVE SQL PATTERNS — Study, then write your own in db/schema.sql ──

# Pattern 1: Table with UUID primary key and server-side defaults
example_devices_ddl = """
CREATE TABLE IF NOT EXISTS example_table (
    id            UUID         PRIMARY KEY DEFAULT gen_random_uuid(),
    name          VARCHAR(64)  UNIQUE NOT NULL,
    created_at    TIMESTAMPTZ  NOT NULL DEFAULT NOW(),
    is_active     BOOLEAN      NOT NULL DEFAULT TRUE
);
"""

# Pattern 2: Table with BIGSERIAL PK, foreign key, and CHECK constraint
example_events_ddl = """
CREATE TABLE IF NOT EXISTS example_events (
    id          BIGSERIAL    PRIMARY KEY,
    table_id    VARCHAR(64)  NOT NULL
                REFERENCES example_table(name) ON DELETE CASCADE,
    event_type  VARCHAR(32)  NOT NULL,
    severity    VARCHAR(16)  NOT NULL
                CHECK (severity IN ('LOW', 'MEDIUM', 'HIGH')),
    timestamp   TIMESTAMPTZ  NOT NULL
);
"""

# Pattern 3: Composite index on two columns
example_index_ddl = """
CREATE INDEX IF NOT EXISTS idx_events_table_time
    ON example_events (table_id, timestamp DESC);
"""

print("Study these patterns, then fill in db/schema.sql")
print("Your schema needs: devices, telemetry_logs, alerts + 3 indexes")

## Section 4: The Complex Query — 24-Hour Alert Frequency by Geo-Grid

This is the most important query in the system. It powers the safety analytics dashboard.

### The Business Question

> *"In the past 24 hours, which locations on UC Berkeley campus had the most
> dangerous obstacle collisions?"*

### Geographic Grid Concept

GPS coordinates have too much precision for heatmap grouping:
- `37.871945` and `37.871952` are only 0.7 meters apart — they're the same location for our purposes

Solution: **Round coordinates to 2 decimal places** (≈ 1.1km resolution):
```
ROUND(gps_lat::numeric, 2) → 37.87   ← all points in the same ~1.1km cell
ROUND(gps_lon::numeric, 2) → -122.26
```

### Query Structure (annotated)

```sql
SELECT
    ROUND(gps_lat::numeric, 2)  AS geo_grid_lat,   -- Grid cell lat
    ROUND(gps_lon::numeric, 2)  AS geo_grid_lon,   -- Grid cell lon
    alert_type,
    COUNT(*)                    AS alert_count      -- Events in this cell
FROM alerts
WHERE
    timestamp  > NOW() - INTERVAL '24 hours'        -- Time window
    AND gps_lat BETWEEN 37.85 AND 37.90             -- Berkeley campus lat
    AND gps_lon BETWEEN -122.28 AND -122.22         -- Berkeley campus lon
    AND severity IN ('HIGH', 'CRITICAL')            -- Significant events only
GROUP BY
    ROUND(gps_lat::numeric, 2),
    ROUND(gps_lon::numeric, 2),
    alert_type
ORDER BY alert_count DESC;
```

### Parameterized version (for use with SQLAlchemy `text()`)

In `app/services/alert_service.py`, replace hardcoded values with named parameters:
```sql
WHERE timestamp > :time_window_start
  AND gps_lat BETWEEN :lat_min AND :lat_max
  AND gps_lon BETWEEN :lon_min AND :lon_max
```
Pass them as a dict: `db.execute(sql, {"time_window_start": ..., "lat_min": ..., ...})`

## Section 5: The Microsoft Geolife Dataset

For Lesson 2, you will import **real GPS trajectory data** into your telemetry_logs table.

### Dataset Information

| Property | Value |
|----------|-------|
| **Name** | Microsoft Research Asia — Geolife GPS Trajectories |
| **Download** | https://www.microsoft.com/en-us/research/project/geolife-building-social-networks-using-human-location-history/ |
| **Size** | ~298 MB compressed, 17,621 trajectories, 182 users |
| **Coverage** | Primarily Beijing, China (2007–2012) |

### .plt File Format

Each file contains one GPS trajectory. Lines 1–6 are headers (ignore them).
Line 7 onward is data, comma-separated:

```
Geolife GPS track
WGS 84
Altitude is in Feet
Reserved 3
0,2,255,My Tracks,0,0,2,8421376
0
39.906631,116.385564,0,492,40097.5864583333,2009-10-11,14:04:29   ← data starts here
39.906611,116.385544,0,492,40097.5865277778,2009-10-11,14:04:36
```

Column layout (0-indexed):
- `[0]` Latitude
- `[1]` Longitude
- `[2]` Always 0 (ignore)
- `[3]` Altitude in feet (ignore for VisionStream)
- `[4]` Fractional days since 12/30/1899 (ignore — use date+time columns instead)
- `[5]` Date string: `YYYY-MM-DD`
- `[6]` Time string: `HH:MM:SS`

### Download and Setup Steps

```bash
# 1. Download from the URL above (look for the .zip download link)
# 2. Extract to: VisionStream/data/geolife/
# 3. Directory structure should be:
#      data/geolife/Data/000/Trajectory/20081023025304.plt
#      data/geolife/Data/001/Trajectory/...
# 4. Run: python scripts/seed_geolife_data.py
```

## Section 6: Illustrative Code — Batch Insert Pattern

Importing 100,000+ rows requires batch insertion.
**Never** insert one row at a time in a loop — that's 100,000 round-trips to the database.

The pattern below shows how `bulk_insert_mappings()` sends all rows in a single SQL statement.

In [ ]:
# ── ILLUSTRATIVE EXAMPLE ONLY — Study the pattern ──

# Pattern: How to parse a CSV-like file and batch insert rows

# from app.database import SessionLocal
# from app.models.telemetry import TelemetryLog
# from datetime import datetime

# def parse_one_file(filepath: str) -> list[dict]:
#     """Parse a text file and return a list of dicts."""
#     records = []
#     with open(filepath, "r") as f:
#         for i, line in enumerate(f):
#             if i < 6:          # Skip header lines
#                 continue
#             parts = line.strip().split(",")
#             if len(parts) < 7:
#                 continue
#
#             records.append({
#                 "device_id":  "seed-device",
#                 "gps_lat":    float(parts[0]),
#                 "gps_lon":    float(parts[1]),
#                 "timestamp":  datetime.strptime(
#                                   f"{parts[5]} {parts[6]}", "%Y-%m-%d %H:%M:%S"
#                               ),
#                 "alert_type": "NONE",
#             })
#     return records


# def insert_batch(records: list[dict]) -> None:
#     """Insert a batch of records in a SINGLE SQL statement."""
#     db = SessionLocal()
#     try:
#         # bulk_insert_mappings sends ONE INSERT with all rows
#         # This is ~100x faster than calling db.add() in a loop
#         db.bulk_insert_mappings(TelemetryLog, records)
#         db.commit()
#         print(f"Inserted {len(records)} rows")
#     finally:
#         db.close()


# Performance comparison:
comparison = {
    "One INSERT per row (loop)": "100,000 round-trips to DB ≈ 5 minutes",
    "bulk_insert_mappings() (batch 1000)": "100 round-trips to DB ≈ 3 seconds",
    "COPY command (psql)": "1 command ≈ 0.5 seconds (fastest, but requires direct DB access)",
}

for method, speed in comparison.items():
    print(f"{method:45s} → {speed}")

---

## 📌 YOUR TASK — File Map

### Step 1 — Write the Database Schema (`db/schema.sql`)

Open `db/schema.sql` and implement all three `CREATE TABLE` statements:

| Table | Key Columns | Constraints |
|-------|-------------|-------------|
| `devices` | id (UUID PK), device_id (UNIQUE), owner_name, registered_at, is_active | gen_random_uuid(), DEFAULT NOW() |
| `telemetry_logs` | id (BIGSERIAL PK), device_id (FK), timestamp, gps_lat, gps_lon, alert_type | FK ON DELETE CASCADE |
| `alerts` | id (BIGSERIAL PK), device_id (FK), timestamp, severity (CHECK), resolved | CHECK IN ('LOW','MEDIUM','HIGH','CRITICAL') |

Then add the 3 indexes for query performance.

**Test your SQL:**
```bash
# Apply the schema to your local PostgreSQL:
psql -U visionstream -d visionstream -f db/schema.sql
```

### Step 2 — Write the Complex Query (`db/schema.sql`)

At the bottom of `db/schema.sql`, write the 24-hour geo-grid alert frequency query.
Then copy it into `app/services/alert_service.get_alert_stats()` as a `text()` query.

### Step 3 — Implement ORM Models (`app/models/`)

| File | Class | What to implement |
|------|-------|-------------------|
| `app/models/device.py` | `Device` | All columns, 2 relationships (to TelemetryLog, Alert) |
| `app/models/telemetry.py` | `TelemetryLog` | All columns, relationship back to Device |
| `app/models/alert.py` | `Alert` | All columns, CHECK constraint in `__table_args__`, relationship |

### Step 4 — Database Connection (`app/database.py`)

Implement:
- `engine` with connection pooling (pool_size=10, max_overflow=20)
- `SessionLocal` factory
- `get_db()` generator (yield + finally close)

### Step 5 — Seed the Database (`scripts/seed_geolife_data.py`)

Implement all three functions:
1. `parse_plt_file()` — parse one .plt file, yield dicts
2. `batch_insert_telemetry()` — bulk insert a list of dicts
3. `seed_database()` — orchestrate the full import

```bash
# Run the seed script:
python scripts/seed_geolife_data.py

# Verify rows were inserted:
# psql> SELECT COUNT(*) FROM telemetry_logs;
```

### Step 6 — Implement `alert_service.get_alert_stats()`

In `app/services/alert_service.py`, implement `get_alert_stats()` using the SQL query
you wrote in Step 2. Use `db.execute(text(...), {...})` to run the parameterized query.

---

## ✅ Lesson 2 Self-Assessment Checklist

- [ ] `db/schema.sql` creates all 3 tables without errors in psql
- [ ] All 3 indexes are created in `db/schema.sql`
- [ ] The complex geo-grid aggregation query returns results on seeded data
- [ ] `Device`, `TelemetryLog`, `Alert` ORM models match the SQL schema
- [ ] ORM relationships work: `device.telemetry_logs` and `device.alerts` are accessible
- [ ] `app/database.py` `get_db()` is implemented (yields session, closes in finally)
- [ ] `python scripts/seed_geolife_data.py` imports at least 10,000 rows
- [ ] `SELECT COUNT(*) FROM telemetry_logs;` shows rows in psql
- [ ] `GET /alerts/stats` endpoint returns data when alerts exist
- [ ] GitHub commit pushed: `feat(db): complete Lesson 2 schema and seed script`

---

**When all boxes are checked → you are ready for Lesson 3.**